### Practice: Parameter Efficient Fine-Tuning
In this notebook, you're gonna fine-tune large language models within limited GPU memory.

In [1]:
%pip install --quiet transformers==4.34.1 accelerate==0.24.0 sentencepiece==0.1.99 optimum==1.13.2 peft==0.5.0 bitsandbytes==0.41.2.post2

import torch
import torch.nn as nn
import torch.nn.functional as F

import transformers
from tqdm.auto import tqdm, trange
assert torch.cuda.is_available(), "you need cuda for this part"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Example part

In [ ]:
model_name = 'Enoch/llama-7b-hf'

# loading Llama tokenizer ...
tokenizer = transformers.LlamaTokenizer.from_pretrained(model_name, device_map=device)
tokenizer.pad_token_id = tokenizer.eos_token_id

# ... and the model itself
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto',
    low_cpu_mem_usage=True,
    offload_state_dict=True,
    load_in_4bit=True,
    torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
)
for param in model.parameters():
    param.requires_grad=False

model.gradient_checkpointing_enable()  # only store a small subset of activations, re-compute the rest.
model.enable_input_require_grads()     # override an implementation quirk in gradient checkpoints that disables backprop unless inputs require grad
# more on gradient checkpointing: https://pytorch.org/docs/stable/checkpoint.html https://arxiv.org/abs/1604.06174

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thouroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


Loading checkpoint shards:   0%|          | 0/33 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:484: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(checkpoint_file, map_location=map

### Prompt tuning: the story of a fox

![img](https://i.imgur.com/Ux3qQAu.png) (source: theodd1souts.fandom.com)

In [ ]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(10):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0].cpu().numpy().tolist()))


Output: <s>A quick brown fox jumps over the lazy dog.
A quick


What a blatant lie! This particular fox assures you that it didn't in fact jump over the lazy dog. No, sir! The fox was just minding its own business. __Your task is to train the model to say truth: no dog was jumped over today.__

In [ ]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
outputs = model(**batch)

next_word_logits = outputs.logits[:, :-1]
true_next_tokens = batch['input_ids'][:, 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

print("Loss:", loss)

Loss: tensor(3.0725, device='cuda:0', grad_fn=<NllLossBackward0>)


Except, we can't train the entire model - that would be 28GB gradients in float32. Instead, let's run [prompt tuning](https://arxiv.org/abs/2104.08691).

![img](https://i.imgur.com/VwNNKnb.png)


In [ ]:
class WordEmbeddingsWithLearnedPrompts(nn.Module):
    """
    To perform prompt tuning, you will need to replace the model's original word embeddings with a layer - THIS layer
    - that inserts trainable prompts instead of the first N token embeddings.
    """

    def __init__(self, word_embeddings: nn.Embedding, num_prompts: int):
        super().__init__()
        self.original_word_embeddings = word_embeddings
        self.num_prompts = num_prompts
        self.learnable_prompts = nn.Parameter(
            torch.randn(1, num_prompts, word_embeddings.embedding_dim), requires_grad=True
        )

    def forward(self, input_ids: torch.LongTensor):
        # input_ids shape: [batch_size, seq_length]
        assert input_ids.dtype == torch.int64
        assert input_ids.shape[1] > self.num_prompts
        assert torch.all(input_ids[:, :self.num_prompts] == tokenizer.pad_token_id).item(), (
            "Don't forget to prepend several BOS tokens to input_ids"
        )

        # Embed the input_ids using the original word embeddings
        input_embeddings = self.original_word_embeddings(input_ids)  # Shape: [batch_size, seq_length, embedding_dim]

        # Replace the first num_prompts token embeddings with the learnable prompts
        batch_size = input_ids.shape[0]
        learnable_prompts_expanded = self.learnable_prompts.expand(batch_size, -1, -1)  # Shape: [batch_size, num_prompts, embedding_dim]
        remaining_embeddings = input_embeddings[:, self.num_prompts:, :]  # Shape: [batch_size, seq_length - num_prompts, embedding_dim]

        # Concatenate learnable prompts with the embeddings of the remaining tokens
        output_embeddings = torch.cat([learnable_prompts_expanded, remaining_embeddings], dim=1)

        return output_embeddings


In [ ]:
num_prompts = 16
test_emb_layer = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)
test_input_ids = tokenizer("a cat say on a may", return_tensors='pt')['input_ids'].to(device)

space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
test_inputs_with_prompts = torch.cat([space_for_prompts, test_input_ids], dim=1)

with torch.cuda.amp.autocast():
  test_prompt_embeddings = test_emb_layer(test_inputs_with_prompts)

assert test_prompt_embeddings.shape[:2] == test_inputs_with_prompts.shape
assert test_prompt_embeddings.shape[-1] == model.config.hidden_size
assert torch.allclose(test_prompt_embeddings[:, :num_prompts], test_emb_layer.learnable_prompts.float())
assert torch.allclose(test_prompt_embeddings[:, num_prompts:], model.model.embed_tokens(test_input_ids).float())
print("Looks legit!")

Looks legit!


<ipython-input-18-acb6260f9ffb>:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


__Now that it works,__ let's inject learnable prompts into the main model and teach it about foxes.

In [ ]:
assert isinstance(model.model.embed_tokens, nn.Embedding), "you have already replaced the embedding layer. If the replacement is broken, please reload the model"

model.model.embed_tokens = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)

opt = torch.optim.Adam([model.model.embed_tokens.learnable_prompts], lr=0.01)

In [ ]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)

# Define optimizer for the learnable prompts
opt = torch.optim.Adam([model.model.embed_tokens.learnable_prompts], lr=0.01)

# Training loop
num_epochs = 100  # Maximum number of epochs to train
loss_threshold = 0.1  # Desired loss value

for epoch in range(num_epochs):
    # Forward pass
    outputs = model(**batch)
    next_word_logits = outputs.logits[:, num_prompts : -1, :]  # Exclude prompt logits and last position
    true_next_tokens = batch['input_ids'][:, num_prompts + 1:]  # Exclude prompt tokens and shift by one

    # Compute loss
    loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

    # Backward pass and optimization
    opt.zero_grad()
    loss.backward()
    opt.step()

    # Print loss for monitoring
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss.item()}")

    # Early stopping condition
    if loss.item() <= loss_threshold:
        print("Loss threshold reached. Stopping training.")
        break
else:
    print("Maximum epochs reached without meeting loss threshold.")


Epoch 1/100, Loss: 7.143682956695557
Epoch 2/100, Loss: 6.534489154815674
Epoch 3/100, Loss: 6.012500286102295
Epoch 4/100, Loss: 5.558838844299316
Epoch 5/100, Loss: 5.177173614501953
Epoch 6/100, Loss: 4.839608192443848
Epoch 7/100, Loss: 4.5710530281066895
Epoch 8/100, Loss: 4.33101224899292
Epoch 9/100, Loss: 4.1062798500061035
Epoch 10/100, Loss: 3.8896265029907227
Epoch 11/100, Loss: 3.6745431423187256
Epoch 12/100, Loss: 3.4574363231658936
Epoch 13/100, Loss: 3.238452196121216
Epoch 14/100, Loss: 3.019965171813965
Epoch 15/100, Loss: 2.8044984340667725
Epoch 16/100, Loss: 2.5935888290405273
Epoch 17/100, Loss: 2.3884737491607666
Epoch 18/100, Loss: 2.1919004917144775
Epoch 19/100, Loss: 2.0083112716674805
Epoch 20/100, Loss: 1.840155839920044
Epoch 21/100, Loss: 1.6829971075057983
Epoch 22/100, Loss: 1.5284016132354736
Epoch 23/100, Loss: 1.374525547027588
Epoch 24/100, Loss: 1.2294597625732422
Epoch 25/100, Loss: 1.1005393266677856
Epoch 26/100, Loss: 0.9856714010238647
Epoch 2

In [ ]:
# Final loss assertion
assert loss.item() <= loss_threshold, "Training did not reduce loss to the desired threshold."
print("Good job!")

Good job!


In [ ]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)


for i in range(15):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0, num_prompts:].cpu().numpy().tolist()))

# if you did everything right, the model will deny that the fox jumped over the lazy dog


Output: <s>A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it


### Using HuggingFace PEFT (2 points)

[`peft`](https://huggingface.co/docs/peft/index) is a transformer's sister library that allows you to apply various __p__arameter __e__fficient __f__ine-__t__uning methods to pre-trained transformers. The library imlements both prompt tuning, prefix tuning, as well as several adapter-based techniques under a common interface:



In [ ]:
import peft
assert isinstance(model.model.embed_tokens, nn.Embedding), "please reload the model"

peft_config = peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=16)
model = peft.get_peft_model(model, peft_config)  # note: for most peft methods, this line also modifies model in-place
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total parameters (excluding quantization):", sum(p.numel() for p in model.parameters()))

Trainable parameters: 65536
Total parameters (excluding quantization): 3500478464


In [ ]:
# Your task: optimize the PEFT-wrapped model to achieve next token prediction loss < 0.1, but this time using PEFT
# Please note: you no longer need to prepend PAD tokens, but you still need to skip :num_virtual_tokens: first logits.
# Finally, generate the sentence to make sure that the model learned the truth.

In [ ]:
# Training Configuration
num_epochs = 100  # Max number of epochs
loss_threshold = 0.1  # Desired loss
learning_rate = 0.01  # Learning rate for the optimizer

# Define the optimizer for trainable parameters (PEFT prompts)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [ ]:
# Define the ground truth sentence
# the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors="pt", return_token_type_ids=False).to(device)

In [ ]:
# Training Configuration
num_epochs = 100  # Max number of epochs
loss_threshold = 0.1  # Desired loss threshold
learning_rate = 0.01  # Learning rate

# Define the optimizer for trainable parameters (PEFT prompts)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Define the ground truth
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors="pt", return_token_type_ids=False).to(device)

# Training Loop
for epoch in range(num_epochs):
    # Forward pass
    outputs = model(**batch)

    # Skip logits for virtual tokens and the last token
    next_word_logits = outputs.logits[:, peft_config.num_virtual_tokens:-1, :]  # Skip virtual tokens
    true_next_tokens = batch['input_ids'][:, 1:]  # Shift ground truth tokens by one

    # Compute the loss
    loss = F.cross_entropy(
        next_word_logits.reshape(-1, next_word_logits.size(-1)),  # Flatten logits
        true_next_tokens.reshape(-1)  # Flatten ground truth tokens
    )

    # Backpropagation
    optimizer.zero_grad()  # Reset gradients
    loss.backward()  # Compute gradients
    optimizer.step()  # Update trainable parameters (PEFT prompts)

    # Print loss for tracking
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss.item()}")

    # Stop training if loss is below threshold
    if loss.item() < loss_threshold:
        print("Loss threshold reached. Stopping training.")
        break
else:
    print("Maximum epochs reached without meeting the loss threshold.")

Epoch 1/100, Loss: 7.675178050994873
Epoch 2/100, Loss: 6.892231464385986
Epoch 3/100, Loss: 6.3485493659973145
Epoch 4/100, Loss: 5.904366970062256
Epoch 5/100, Loss: 5.507501125335693
Epoch 6/100, Loss: 5.568404197692871
Epoch 7/100, Loss: 5.064216136932373
Epoch 8/100, Loss: 4.949523448944092
Epoch 9/100, Loss: 4.826657772064209
Epoch 10/100, Loss: 4.69673490524292
Epoch 11/100, Loss: 4.5657548904418945
Epoch 12/100, Loss: 4.434375286102295
Epoch 13/100, Loss: 4.301575660705566
Epoch 14/100, Loss: 4.167079448699951
Epoch 15/100, Loss: 4.031871318817139
Epoch 16/100, Loss: 3.897719383239746
Epoch 17/100, Loss: 3.766268730163574
Epoch 18/100, Loss: 3.6382246017456055
Epoch 19/100, Loss: 3.512860059738159
Epoch 20/100, Loss: 3.387976884841919
Epoch 21/100, Loss: 3.2605791091918945
Epoch 22/100, Loss: 3.128392219543457
Epoch 23/100, Loss: 2.991361141204834
Epoch 24/100, Loss: 2.8520336151123047
Epoch 25/100, Loss: 2.7146761417388916
Epoch 26/100, Loss: 2.5834615230560303
Epoch 27/100, L

In [ ]:
# Final assertion to ensure loss is below threshold
assert loss.item() < loss_threshold, "Training failed to reduce loss below threshold."
print("Training successful! Loss is below 0.1.")

Training successful! Loss is below 0.1.


In [ ]:
prompt = "A quick brown fox"
batch = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)

# Generate 15 tokens
for i in range(15):
    # Forward pass to get the logits
    outputs = model(**batch)
    next_token = outputs.logits[0, -1].argmax(-1).reshape(1, 1)

    # Append the next token to input_ids
    batch["input_ids"] = torch.cat([batch["input_ids"], next_token], dim=-1)

    # Update the attention_mask to match the new input_ids length
    new_attention_mask = torch.ones_like(next_token, dtype=batch["attention_mask"].dtype).to(device)
    batch["attention_mask"] = torch.cat([batch["attention_mask"], new_attention_mask], dim=-1)

# Decode the generated sequence
# Skip the virtual tokens (if applicable) by slicing `batch["input_ids"][:, num_prompts:]`
decoded_output = tokenizer.decode(batch["input_ids"][0].cpu().numpy().tolist(), skip_special_tokens=True)
print("\nOutput:", decoded_output)



Output: A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it


### Parameter-efficient finetuning with LoRA (2 points)

When training on more serious tasks, you can use low-rank adapters based on the [LoRA paper](https://arxiv.org/pdf/2106.09685.pdf).

The core idea is to add low-rank adapters __in parallel with existing linear layers,__ like this:
<center><img src="https://i.imgur.com/6bQLNiG.png" width=240px></center>

In the original LoRA paper, the adapters were only added to attention projection matrices. However, [subsequent works](https://arxiv.org/abs/2305.14314) show that it is useful to adapt FFNs as well. But before we do any training, we need to implement the basic LoRA layer.

In [ ]:
# re-load the model to remove any previous PEFT tuners
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, device_map='auto', low_cpu_mem_usage=True, offload_state_dict=True,
    load_in_4bit=True, torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
)
for param in model.parameters():
    param.requires_grad=False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

Loading checkpoint shards:   0%|          | 0/33 [00:00<?, ?it/s]

In [ ]:
class LoRALayer(nn.Module):
    """Wraps a linear layer with LoRA-like adapter. Wraps an existing OPT linear layer"""
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module  # pre-trained (frozen) linear layer
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))
        nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))

    def forward(self, input):
        # Apply self.module and LoRA adapter, return the sum (self.module outputs + adapter outputs)
        original_output = self.module(input)
        lora_output = input @ self.adapter_A @ self.adapter_B

        return original_output + lora_output

In [ ]:
# test your implementation
test_linear = nn.Linear(128, 128)
test_linear.weight.data[...] = torch.eye(128)
test_adapter = LoRALayer(test_linear, rank=8)

assert torch.allclose(test_adapter(torch.ones(1, 1, 128)), test_linear.bias + 1), "please check your forward pass"

test_adapter.adapter_A.data[...] = torch.linspace(0.1, -0.5, 128 * 8).view(128, 8)
test_adapter.adapter_B.data[...] = torch.linspace(0.5, -0.1, 128 * 8).view(8, 128)
test_linear.bias.data[...] = torch.linspace(1., -1., 128)

dummy_loss = F.mse_loss(test_adapter(torch.ones(1, 128) / 128).squeeze(), torch.linspace(-1, 1, 128))
assert torch.allclose(dummy_loss, torch.tensor(1.3711389), rtol=0, atol=1e-4)
dummy_loss.backward()
assert all(w.grad is not None for w in [test_adapter.adapter_A, test_adapter.adapter_B]), "some adapter weights have no grad"
assert torch.allclose(test_adapter.adapter_A.grad.sum(), torch.tensor(-0.60158), rtol=0, atol=1e-4), "bad grad w.r.t. A"
assert torch.allclose(test_adapter.adapter_B.grad.sum(), torch.tensor(0.9931), rtol=0, atol=1e-4), "bad grad w.r.t. B"
# note: bad grad means that your code is different from LoRA paper OR that your code is not autograd-friendly (e.g. no_grad)
del dummy_loss, test_linear, test_adapter
print("All tests passed!")

All tests passed!


### Apply LoRA to the model

The code below applies LoRA adapters on top of Q/K/V linear layers in Llama attention. You may also choose to modify other layers:
* self_attn.o_proj - attention output projection
* mlp.up_proj, mlp.gate_proj, mlp.down_proj - transformer feedforward layers
* lm_head - output LM head

__Note:__ please scroll down for the homework task

In [ ]:
lora_rank = 8

for name, module in model.model.layers.named_modules():
    if 'LlamaDecoderLayer' in repr(type(module)):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)

assert sum(isinstance(module, LoRALayer) for module in model.modules()) == 96  # for Llama-7B

NameError: name 'model' is not defined

In [ ]:
batch = tokenizer("This model wants to share its greatest secret:", return_tensors='pt', return_token_type_ids=False)
# test a single training step, make sure we get meaningful gradients
with torch.cuda.amp.autocast(dtype=torch.float32):
    out = model.forward(**batch)
    (out.logits.norm() / 100).backward()

for i, module in enumerate(model.modules()):
    if isinstance(module, LoRALayer):
        assert module.adapter_B.grad is not None
        assert module.adapter_B.grad.norm().item() > 0

model.zero_grad(set_to_none=True)
print("Grad check successful, well done!")

NameError: name 'tokenizer' is not defined

### (example) How to train your model

The example below shows how to train the LoRA adapters on a dummy dataset. You will need to run a _similar_ training task later.

__Note:__ please scroll down for the homework task

In [ ]:
# checking if the model can learn. Change max_steps for proper training
import datasets
data = datasets.load_dataset("Abirate/english_quotes", split="train[:32]") # 32 lines
data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)
model._hf_peft_config_loaded = True  # silence a warning from HF trainer

trainer = transformers.Trainer(
    model=model, train_dataset=data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=1,
        # note: if you want larger batch size, increase gradient_accumulation_steps
        warmup_steps=250, max_steps=100, learning_rate=2e-4, fp16=True,
        logging_steps=1, output_dir='outputs', report_to=None),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
# if you see cache warnings, set `model.config.use_cache = False` to silence them. Please re-enable for inference!

trainer.train()

# NOTE: this is just an example! you do not have to wait for this progressbar to finish :)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss
1,1.891200
2,1.696000
3,0.896900
4,1.744600
5,1.168000
6,0.730000
7,1.525700
8,1.063700
9,0.669600
10,1.427400


TrainOutput(global_step=100, training_loss=0.5410390722751618, metrics={'train_runtime': 150.7473, 'train_samples_per_second': 1.327, 'train_steps_per_second': 0.663, 'total_flos': 621258424123392.0, 'train_loss': 0.5410390722751618, 'epoch': 6.25})

# Homework: *actually* train the model (10 points)

Your task is to fine-tune the model to _generate python code_. Please use the above examples for inspiration. More specifically,

* __dataset:__ use [codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) or any other data containing python code. Since you do not need much data for this excercise, it is enough to use just shorter train subset of `codeparrots`
* __preprocessing:__ select python code based on file extentions (.py)  (may skip in case of codeparrot - it is 100% python)
* __short lines:__ please take the first 512 characters of each line
* __adapter type:__ please use LoRA as defined above __plus at least one of:__
   - extra adapter on lm_head
   - extra adapter on MLP components (mlp.*)
   - trainable input embeddings (requires tweaking memory usage)

* __training:__ you do not have to train to convergence. If all goes well, your model should `.generate` code after 500 steps. Please use batch size of at least 4 (4 x 1 x 512 tokens) using `gradient_accumulation_steps=4`.


Note: the peft library also has LoRA implementation. However, we ask that for this assignment you show at least one complete training run with your own LoRA code.

__Alternative assignment:__ Instead of doing python code, feel free to substitute the task with any other dataset, e.g. your favorite artist or podcast, as long as it's ethical. If you choose your own task, please show examples of what your model learned - or did not learn, akin to the code examples below.

In [2]:
from datasets import load_dataset, Dataset
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader

from transformers import Trainer, TrainingArguments


/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [3]:
def generate_text(model, prompt:str, seq_len:int=50):
    '''Функция для генерации текста на основе промпта

    :model: model for generate sequance
    :prompt: prompt for generate sequance
    :seq_len: len of generated sequance

    :return: generated sequance len = seq_len

    '''
    batch = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)

    # Generate tokens
    for i in range(seq_len):
        # Forward pass to get the logits
        outputs = model(**batch)
        next_token = outputs.logits[0, -1].argmax(-1).reshape(1, 1)

        # Append the next token to input_ids
        batch["input_ids"] = torch.cat([batch["input_ids"], next_token], dim=-1)

        # Update the attention_mask to match the new input_ids length
        new_attention_mask = torch.ones_like(next_token, dtype=batch["attention_mask"].dtype).to(device)
        batch["attention_mask"] = torch.cat([batch["attention_mask"], new_attention_mask], dim=-1)

    decoded_output = tokenizer.decode(batch["input_ids"][0].cpu().numpy().tolist(), skip_special_tokens=True)

    return decoded_output

In [4]:
class LoRALayer(nn.Module):
    """Wraps a linear layer with LoRA-like adapter. Wraps an existing OPT linear layer"""
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module  # pre-trained (frozen) linear layer
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))
        nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))

    def forward(self, input):
        # Apply self.module and LoRA adapter, return the sum (self.module outputs + adapter outputs)
        original_output = self.module(input)
        lora_output = input @ self.adapter_A @ self.adapter_B

        return original_output + lora_output

In [5]:
def combine_lines(lines, max_lengh=512):
    '''Функция соединяет строку с следующей, если их суммарная длинна не превышает 512 слов.

    Иначе запоминаем текущую строку и начинаем работать со следующей, так же добавляя к ней последующие'''
    combined = []
    current_line = ""
    for line in lines:
        # Добавляем новую строку к текущей
        new_line = current_line + "\n" + line if current_line else line
        # Токенизируем новую строку
        #tokenized = tokenizer(new_line, truncation=False, return_tensors="pt") -- очень долго
        #if len(tokenized['input_ids'][0]) <= max_lengh:
        if len(new_line) <= max_lengh:
            current_line = new_line
        else:
            # Если превысили лимит, добавляем текущую строку в список
            combined.append(current_line)
            current_line = line
    # Добавляем последнюю строку
    if current_line:
        combined.append(current_line)
    return combined

In [6]:
prompts =  ['', 'import', 'from', 'while', 'try', 'if', 'for', 'torch', 'def', 'assert', '!pip']  # feel free to add a few more that are not 100% assiciated with Python

# <A WHOLE LOT OF YOUR CODE>
# generate baseline samples with the selected prompts before finetuning
# please feel free to use transformers.Trainer (as above) or your custom training code
# after the training concludes, please show examples of text generated by your model. It is expected to look like Python code fragments
# print the generation examples nicely (suggestion: use pandas or HTML) for easier comparison
# note: your LoRA-enhanced model can run generation the same way as the non-trained model (above)

___Load model___

In [7]:
model_name = 'Enoch/llama-7b-hf'

# loading Llama tokenizer ...
tokenizer = transformers.LlamaTokenizer.from_pretrained(model_name, device_map=device)
tokenizer.pad_token_id = tokenizer.eos_token_id

# load the model
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, device_map='auto', low_cpu_mem_usage=True, offload_state_dict=True,
    load_in_4bit=True, torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
)

# Freeze model
for param in model.parameters():
    param.requires_grad=False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thouroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Loading checkpoint shards:   0%|          | 0/33 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:484: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(checkpoint_file, map_location=map

In [8]:
# Model from HF
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )


In [9]:
# This template helps to compare generated code samples in pretty table form
# feel free to present your work in other forms

from IPython.display import HTML, display
table_template = """<table style="border:1px solid black" >
  <tr>
    <th style="text-align: center; border:1px solid black">PROMPT</th>
    <th style="text-align: center; border:1px solid black">BEFORE</th>
    <th style="text-align: center; border:1px solid black">AFTER</th>
  </tr>
{}
</table>"""

row_template = '''  <tr>
    <td style="width:20%; border:1px solid black"><pre align="left">`{}`</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
  </tr>'''

rows = []
memory_out_pretrained_model = []

for prompt in prompts:

    out_pretrained_model = generate_text(model=model, prompt=prompt, seq_len=100)
    memory_out_pretrained_model.append(out_pretrained_model)

    rows.append(row_template.format(prompt, out_pretrained_model, "TO BE GENERATED AFTER FINETUNING"))

display(HTML(table_template.format('\n'.join(rows))))

PROMPT,BEFORE,AFTER
``,▶▶ 2019-2020 School Year The 2019-2020 school year is here! We are so excited to welcome our new students and families to the school. We are also excited to welcome back our returning families. We are looking forward to another great year at the school. We are also looking forward to another great year of learning and growing together. We are also looking forward to another great year of learning and growing together,TO BE GENERATED AFTER FINETUNING
`import`,import Foundation public extension NSURL { public var absoluteString: String { return String(cString: CFBundleGetBundleWithURL(self).UTF8String) } } package com.google.gerrit.server.restapi; import static javax.ws.rs.core.MediaType.APPLICATION_JSON; import com.google.gerrit.extensions.restapi.RestApiModule; import,TO BE GENERATED AFTER FINETUNING
`from`,from __future__ import absolute_import from __future__ import division from __future__ import print_function import os import sys from absl import flags from tensorflow.python import pywrap_tensorflow from tensorflow.python.eager import context from tensorflow.python.eager import function from tensorflow.python.eager import test from tensorflow.python.eager import backprop from tensorflow.python.e,TO BE GENERATED AFTER FINETUNING
`while`,"while(1) while(1) { // do something } \end{code} Comment: This is not the same as the OP's code. Comment: @Jeffrey: It's the same as the OP's code, except that it's not a function. Comment: @Jeffrey: The OP's code is a function, but it's not a function declaration. Comment: @",TO BE GENERATED AFTER FINETUNING
`try`,try to find the best solution for your needs. We are a team of professionals with a long experience in the field of web development. We are a team of professionals with a long experience in the field of web development. We are a team of professionals with a long experience in the field of web development. We are a team of professionals with a long experience in the field of web development. We are a team of professionals with a long experience in the field of web development,TO BE GENERATED AFTER FINETUNING
`if`,"if ( !window.atmosphere ) { window.atmosphere = {}; } (function () { var o = atmosphere.util, atmosphere = atmosphere.atmosphere = function () { var _isClosed = false, _isOpening = false, _isOpen = false, _isClosing = false, _isError = false, _isReady =",TO BE GENERATED AFTER FINETUNING
`for`,for the 2019-2020 school year. The application process for the 2019-2020 school year is now open. The application process for the 2019-2020 school year is now open. Please click here to apply. The application process for the 2019-2020 school year is now open. Please click here to apply. The application process for the 2,TO BE GENERATED AFTER FINETUNING
`torch`,"torchbearer 2017-05-18 19:55:25 UTC #1 I’m a newbie to the world of RPGs, and I’m looking for a game that I can play with my wife. We’re both in our 30s, and we’re looking for a game that we can play together. We’re both new to the world of RPGs, and we’re looking",TO BE GENERATED AFTER FINETUNING
`def`,"def _test_get_version(self): """""" Tests the get_version function """""" self.assertEqual(self.get_version(), ""0.1"") class TestGetVersion(unittest.TestCase): """""" Tests the get_version function """""" def test_get_version(self): """""" Tests the get_version function """""" self",TO BE GENERATED AFTER FINETUNING
`assert`,assert(isPrimitive(void 0)); assert(isPrimitive(void 1)); assert(isPrimitive(void 2)); assert(isPrimitive(void 3)); assert(isPrimitive(void 4)); assert(isPrimitive(void 5)); assert(isPrimitive(void 6)); assert(isPrimitive(void 7)); assert(isPrimitive(void 8)); assert(,TO BE GENERATED AFTER FINETUNING


___Add LoRa layers to model___

In [10]:
# Change W_q, W_k, W_v in attention layer
lora_rank = 8

for name, module in model.model.layers.named_modules():
    if 'LlamaDecoderLayer' in repr(type(module)):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)

assert sum(isinstance(module, LoRALayer) for module in model.modules()) == 96  # for Llama-7B


# Change head layer
model.lm_head = LoRALayer(model.lm_head, rank=lora_rank).to(device)


# Change layers in ffn
for name, module in model.model.layers.named_modules():
    if 'LlamaDecoderLayer' in repr(type(module)):
        module.mlp.gate_proj = LoRALayer(module.mlp.gate_proj, rank=lora_rank).to(device)
        module.mlp.up_proj = LoRALayer(module.mlp.up_proj, rank=lora_rank).to(device)
        module.mlp.down_proj = LoRALayer(module.mlp.down_proj, rank=lora_rank).to(device)

In [11]:
# New model
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): LoRALayer(
            (module): Linear4bit(in_features=4096, out_features=4096, bias=False)
          )
          (k_proj): LoRALayer(
            (module): Linear4bit(in_features=4096, out_features=4096, bias=False)
          )
          (v_proj): LoRALayer(
            (module): Linear4bit(in_features=4096, out_features=4096, bias=False)
          )
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): LoRALayer(
            (module): Linear4bit(in_features=4096, out_features=11008, bias=False)
          )
          (up_proj): LoRALayer(
            (module): Linear4bit(in_features=4096, out_features=11008, bias=False)
          )
       

___Train data___

In [12]:
# Load train data
ds = load_dataset("codeparrot/codeparrot-clean", streaming=True)

ds_train = ds["train"]

train_rows = []

# Split and cat all rows (max 512 chars)
for item in ds_train.take(100000):
  current_code = item['content'].split('\n')

  # cut row
  for row_num in range(len(current_code)):
    current_code[row_num] = current_code[row_num][:512]

  current_code = '\n'.join(current_code)

  train_rows.append(current_code)

Resolving data files:   0%|          | 0/54 [00:00<?, ?it/s]

In [13]:
# Concat lines, maximizing the length to 512 words
train_rows_for_tokenizer = []

for script in tqdm(train_rows):
  train_rows_for_tokenizer.append(combine_lines(script.split('\n')))

  0%|          | 0/100000 [00:00<?, ?it/s]

___Prepaid data___

In [14]:
# tokenize and create Dataset
data = Dataset.from_dict({"code": train_rows})

data = data.map(lambda samples: tokenizer(
                samples['code'],
                truncation=True,
                return_tensors="pt",
                max_length=512,
                padding="max_length"
                ),
                batched=True
                )

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [15]:
# Create batches
batches = []
batch_size = 2

for i in range(0, len(data), batch_size):
    batch = data[i:i + batch_size]
    # input_ids and attention_mask to tensor
    input_ids = torch.stack([torch.tensor(item) for item in batch['input_ids']])
    attention_mask = torch.stack([torch.tensor(item) for item in batch['attention_mask']])
    batches.append({'input_ids': input_ids, 'attention_mask': attention_mask})

In [16]:
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, batches):
        self.data = batches

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        return {
            'input_ids': self.data[idx]['input_ids'].clone().detach(),
            'attention_mask': self.data[idx]['attention_mask'].clone().detach(
        }

train_dataset = CustomDataset(batches)


In [17]:
print(train_dataset[0].keys())

train_dataset[0]['input_ids'].size()

dict_keys(['input_ids', 'attention_mask'])


torch.Size([2, 512])

In [18]:
def custom_collate_fn(batch):
    '''Функция для подготовки батчей данных для модели'''
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])

    input_ids = input_ids.view(-1, input_ids.size(-1))
    attention_mask = attention_mask.view(-1, attention_mask.size(-1))
    labels = input_ids.clone()


    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels,
    }


In [19]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly"
)

In [20]:
model._hf_peft_config_loaded = True  # silence a warning from HF trainer
model.config.use_cache = False
torch.utils.checkpoint.use_reentrant = False

# TrainingArguments
training_args = TrainingArguments(
    output_dir='outputs',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=250,
    max_steps=550,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    report_to="none"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    data_collator=custom_collate_fn,
)

# Learn model
trainer.train()

/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss
10,1.758400
20,1.902700
30,1.556200
40,1.380000
50,1.517700
60,1.142600
70,1.110300
80,0.960000
90,0.930500
100,0.910900


KeyboardInterrupt: 

In [21]:
# Result of fine-tune model

from IPython.display import HTML, display
table_template = """<table style="border:1px solid black" >
  <tr>
    <th style="text-align: center; border:1px solid black">PROMPT</th>
    <th style="text-align: center; border:1px solid black">BEFORE</th>
    <th style="text-align: center; border:1px solid black">AFTER</th>
  </tr>
{}
</table>"""

row_template = '''  <tr>
    <td style="width:20%; border:1px solid black"><pre align="left">`{}`</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
  </tr>'''

rows = []
memory_out_fine_tuned_model = []

for prompt, out_pretrained_model in zip(prompts, memory_out_pretrained_model):

    out_fine_tuned_model = generate_text(model=model, prompt=prompt, seq_len=100)
    memory_out_fine_tuned_model.append(out_fine_tuned_model)

    rows.append(row_template.format(prompt, out_pretrained_model, out_fine_tuned_model))

display(HTML(table_template.format('\n'.join(rows))))

PROMPT,BEFORE,AFTER
``,▶▶ 2019-2020 School Year The 2019-2020 school year is here! We are so excited to welcome our new students and families to the school. We are also excited to welcome back our returning families. We are looking forward to another great year at the school. We are also looking forward to another great year of learning and growing together. We are also looking forward to another great year of learning and growing together,"# Copyright 2015 The TensorFlow Authors. All Rights Reserved. # # Licensed under the Apache License, Version 2.0 (the ""License""); # you may not use this file except in compliance with the License. # You may obtain a copy of the License at # # http://www.apache.org/licenses/LICENSE-2.0 # # Unless required by"
`import`,import Foundation public extension NSURL { public var absoluteString: String { return String(cString: CFBundleGetBundleWithURL(self).UTF8String) } } package com.google.gerrit.server.restapi; import static javax.ws.rs.core.MediaType.APPLICATION_JSON; import com.google.gerrit.extensions.restapi.RestApiModule; import,import os import sys import time import traceback import logging import logging.handlers import logging.config import logging.configurator import logging.root import logging.manager import logging.manager.threading import logging.handlers import logging.handlers.rotating_file_handler import logging.handlers.rotating_file_handler import logging.handlers.rotating_file_handler import logging.hand
`from`,from __future__ import absolute_import from __future__ import division from __future__ import print_function import os import sys from absl import flags from tensorflow.python import pywrap_tensorflow from tensorflow.python.eager import context from tensorflow.python.eager import function from tensorflow.python.eager import test from tensorflow.python.eager import backprop from tensorflow.python.e,"from __future__ import absolute_import import logging import os import sys import time from django.conf import settings from django.core.management.base import BaseCommand from django.utils.translation import ugettext_lazy as _ from zerver.models import UserProfile, get_system_bot from zerver.models import get_client from zerver.models import get_client_descriptor"
`while`,"while(1) while(1) { // do something } \end{code} Comment: This is not the same as the OP's code. Comment: @Jeffrey: It's the same as the OP's code, except that it's not a function. Comment: @Jeffrey: The OP's code is a function, but it's not a function declaration. Comment: @",while True: # read the file f = open(sys.argv[1]) lines = f.readlines() f.close() # split the lines into words words = [w.strip() for w in lines] # sort the words words.sort() # print the words for w in words: print(w) # check if the file is
`try`,try to find the best solution for your needs. We are a team of professionals with a long experience in the field of web development. We are a team of professionals with a long experience in the field of web development. We are a team of professionals with a long experience in the field of web development. We are a team of professionals with a long experience in the field of web development. We are a team of professionals with a long experience in the field of web development,try: from urllib.parse import urlparse except ImportError: from urlparse import urlparse from django.conf import settings from django.core.urlresolvers import reverse from django.test import TestCase from django.utils.http import urlsafe_base64_encode from django.utils.http import urlsafe_base64_decode from django.utils.http import urlsafe_urlencode from django.
`if`,"if ( !window.atmosphere ) { window.atmosphere = {}; } (function () { var o = atmosphere.util, atmosphere = atmosphere.atmosphere = function () { var _isClosed = false, _isOpening = false, _isOpen = false, _isClosing = false, _isError = false, _isReady =",if (typeof require !== 'undefined') { var path = require('path'); var fs = require('fs'); var _ = requi

In [22]:
prompt = 'pd.Dat'
generate_text(model=model, prompt=prompt, seq_len=100)

"pd.DatetimeIndex(['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04', '2013-01-05', '2013-01-06', '2013-01-07', '2013-01-08', '2"

In [26]:
prompt = 'pd.DataFr'
txt = generate_text(model=model, prompt=prompt, seq_len=100)
print(txt)

pd.DataFrames.from_csv(csv_file, header=None, index_col=None,
                         parse_dates=True, infer_datetime_format=True)

# 2017-01-01 00:00:00.000000
# 2017-01-01 00:00:00.000000
# 2


In [25]:
prompt = 'logger'
txt = generate_text(model=model, prompt=prompt, seq_len=100)
print(txt)

logger.debug('Creating new session')
session = Session(request)

# Check if the user is logged in
if session.user is None:
    logger.debug('User not logged in')
    return redirect('/login')

# Check if the user is logged in
if session.user.is_authenticated:
    logger.debug('User is logged in')
    return redirect('/dashboard')

# Check if the user is logged


In [27]:
prompt = 'trainer = Trainer('
txt = generate_text(model=model, prompt=prompt, seq_len=100)
print(txt)

trainer = Trainer(
    config=config,
    data_dir=data_dir,
    model_dir=model_dir,
    model_name=model_name,
    model_version=model_version,
    model_type=model_type,
    model_name_prefix=model_name_prefix,
    model_version_prefix=model_version_prefix,
    model_type_prefix=model_type_prefix,
   


In [28]:
prompt = 'model = CatboostClassifier()'
txt = generate_text(model=model, prompt=prompt, seq_len=500)
print(txt)

model = CatboostClassifier()
model.fit(X, y)

# get the predictions
preds = model.predict(X)

# get the probabilities
prob = model.predict_proba(X)

# get the probabilities of the class with the highest probability
prob_max = prob.argmax()

# get the class with the highest probability
pred = prob[prob_max]

# get the class with the highest probability
pred = prob[pred]

# get the class with the highest probability
pred = prob[pred]

# get the class with the highest probability
pred = prob[pred]

# get the class with the highest probability
pred = prob[pred]

# get the class with the highest probability
pred = prob[pred]

# get the class with the highest probability
pred = prob[pred]

# get the class with the highest probability
pred = prob[pred]

# get the class with the highest probability
pred = prob[pred]

# get the class with the highest probability
pred = prob[pred]

# get the class with the highest probability
pred = prob[pred]

# get the class with the highest probability
pred =

If you reach this: congratulations! you've completed everything in this practice session.

If you want to dig deeper, try to implement prompt-tuning (for bonus points!).
You can read more about prompt tuning variants in paper [1](https://arxiv.org/abs/2104.08691) or paper [2](https://arxiv.org/abs/2101.00190). Both versions can be implemented by passing trainable prompts as `model.forward(..., past_key_values=your_prompts)`.



### Read more

* How post-training quantization works: https://arxiv.org/abs/2208.07339
* An overview of running large models: https://huggingface.co/docs/accelerate/package_reference/big_modeling
* A general library for different adapter types: https://adapterhub.ml/


### [extra info] Running other models.

This notebook's code can run with other models of similar size, such as [Falcon-7B](https://huggingface.co/tiiuae/falcon-7b), [OPT-6.7B](https://huggingface.co/facebook/opt-6.7b) or [BLOOM-7.1B](https://huggingface.co/bigscience/bloom-7b1). However, they will require minor code tweaks:
1. change the model name in `AutoModelForCausalLM.from_pretrained()` __and__ `AutoTokenizer`
2. In the prompt tuning code, change `model.model.embed_tokens` to refer to the target model's word embeddings. Simply `print(model)` to navigate to them.
3. Change code to add Lora layers - specifically where you what the transformer block components, since those components now have different names.